In [1]:
!rm -rf helix_addition && git clone https://github.com/amark-23/helix_addition.git
!pip install -q git+https://github.com/safety-research/circuit-tracer@v0.5.3
!pip install -q -e helix_addition pytest

Cloning into 'helix_addition'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 58 (delta 17), reused 52 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 13.89 KiB | 13.89 MiB/s, done.
Resolving deltas: 100% (17/17), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for helixadd (pyproject.toml) ... done


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [3]:
!cd helix_addition && python -m pytest -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/helix_addition
configfile: pyproject.toml
plugins: jaxtyping-0.3.11, anyio-4.15.1, langsmith-0.14.12, platformdirs-4.13.5, typeguard-4.6.0
collected 16 items                                                             

tests/test_prompts.py::test_zero_shot_tokens PASSED                      [  6%]
tests/test_prompts.py::test_operands_single_token PASSED                 [ 12%]
tests/test_prompts.py::test_few_shot_length PASSED                       [ 18%]
tests/test_prompts.py::test_last_token_is_equals PASSED                  [ 25%]
tests/test_readout.py::test_parse_answer[95-95] PASSED                   [ 31%]
tests/test_readout.py::test_parse_answer[95\n-95] PASSED                 [ 37%]
tests/test_readout.py::test_parse_answer[ 95-95] PASSED                  [ 43%]
tests/tes

In [4]:
!pip show helixadd
!ls helix_addition/src/helixadd

Name: helixadd
Version: 0.1.0
Summary: 
Home-page: 
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Editable project location: /kaggle/working/helix_addition
Requires: 
Required-by: 
clean.py     model.py	 __pycache__  results.py
__init__.py  prompts.py  readout.py


In [5]:
import torch
from helixadd.model import load_model, generate
from helixadd.prompts import make_prompt
from helixadd.readout import parse_answer

model = load_model()
prompts = [make_prompt(36, 59), make_prompt(3, 5), make_prompt(99, 99)]
for p, t in zip(prompts, generate(model, prompts)):
    print(repr(t), parse_answer(t))
print(f"GPU memory peak: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

I0000 00:00:1794160562.430698     842 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
'95\ncalc:' 95
'8\ncalc:' 8
'198\ncalc:' 198
GPU memory peak: 12.3 GB


In [6]:
from helixadd.clean import run_clean

df = run_clean(model, "helix_addition/results/clean_accuracy")
print(f"accuracy: {df.correct.mean():.4f}  ({(~df.correct).sum()} wrong)")
print(df.groupby("carry").correct.mean())

accuracy: 0.9975  (25 wrong)
carry
False    0.997453
True     0.997553
Name: correct, dtype: float64


In [7]:
df[~df.correct]

,a,b,carry,raw,pred,correct,logit_correct
5769,57,74,True,141\ncalc:,141,False,23.796875
7051,70,57,False,137\ncalc:,137,False,25.593750
7052,70,58,False,138\ncalc:,138,False,26.531250
7056,70,62,False,142\ncalc:,142,False,25.187500
7348,73,54,False,137\ncalc:,137,False,23.609375
7349,73,55,False,138\ncalc:,138,False,24.187500
7350,73,56,False,139\ncalc:,139,False,23.078125
7351,73,57,True,140\ncalc:,140,False,24.203125
7352,73,58,True,141\ncalc:,141,False,24.062500
7448,74,54,False,138\ncalc:,138,False,24.359375


In [14]:
from helixadd.clean import run_clean

df0 = run_clean(model, "helix_addition/results/clean_accuracy_zeroshot", few_shot=False)
print(f"accuracy: {df0.correct.mean():.4f}  ({(~df0.correct).sum()} wrong)")
print(df0.groupby("carry").correct.mean())


accuracy: 0.9977  (23 wrong)
carry
False    0.996543
True     0.999110
Name: correct, dtype: float64


In [15]:
set(zip(df0.a[~df0.correct], df0.b[~df0.correct]))

{(0, 91),
 (1, 86),
 (4, 31),
 (12, 34),
 (18, 91),
 (24, 7),
 (47, 70),
 (47, 72),
 (48, 63),
 (48, 70),
 (49, 70),
 (50, 70),
 (60, 64),
 (60, 70),
 (65, 64),
 (70, 64),
 (74, 58),
 (80, 64),
 (80, 79),
 (83, 72),
 (90, 64),
 (95, 94),
 (95, 96)}

In [16]:
!ls helix_addition/results

clean_accuracy	clean_accuracy_zeroshot
